# Certiscan-2D - Vérificateur de conformité des documents présentants un 2D-Code

**Certiscan-2D** est une application qui vérifie l'intégrité et la cohérence des documents officiels sécurisés par un code **2D-Doc**, standard français développé par l'ANTS (Agence Nationale des Titres Sécurisés) et utilisé sur les documents administratifs courants tels que les avis d'imposition, factures d'énergie, justificatifs de ressources...

Le 2D-Doc est un **Data Matrix signé électroniquement** (ECDSA P-256/P-384/P-521). Il encode les données clés du document (nom, montant, émetteur, dates) ainsi qu'une signature cryptographique. La clé privée appartient à l'organisme émetteur et les clés publiques sont distribuées librement via la **TSL (Trusted Service List)** publiée par l'ANTS.

## Etape 1 : Référentiel des types documentaires

L'objectif est de définir les modèles de données utilisés pour l'extraction et la normalisation des documents clients. Ces données sont tirées des spécifications techniques des documents utilisées dans le document suivant, émis par l'Etat Français : [https://ants.gouv.fr/files/1ba15231-0320-40da-819a-655888f43eb9/ants_2d-doc_cabspec_v334.pdf].

On définira donc les différents points de vigilance sur l'étude d'un document, les champs obligatoires et les résultats d'une étude.

In [1]:
from dataclasses import dataclass
from enum import Enum
from typing import Optional


class VerificationStatus(Enum):
    """
    Définition des statuts possibles suite à une étude documentaire.
    """
    VALID = "Document valide - Données cohérentes"
    INVALID = "Échec structurel (non reconnu)"
    CRYPTO_FAIL = "Signature numérique incorrecte"
    SUSPICIOUS = "Incohérence entre données et 2D-Code"


@dataclass
class DocFields:
    """
    Champs communs pour tous les 2D-Codes. Un 2D-Codes n'ayant pas ça sera non conforme
    """
    marqueur_id : str   # marqueur d'identitfication
    version_id : str    # version du 2D-Code
    ca_id : str     # identifiant de l'autorité de certification
    certif_id : str     # identifiant du certificat
    date_emission : str     # date d'émission
    date_signature : str    # date de création de la signature
    code_identification_doc : str   # code d'identification du document
    identifiant_perimetre : Optional[str] = None # code du périmètre à partir de la v03
    pays_emetteur : Optional[str] = None  # pays émetteur du doc


# ================
# On définit ensuite les champs obligatoires pour chaque type de document (donnons 4 exemples)

@dataclass
class JustificatifDomicile(DocFields):
    """Type 00 — Justificatif de domicile générique"""
    adresse_ligne1 : str = None # 10 - Ligne 1 adresse postale
    titre_personne : str = None # 11 - Qualité/titre
    prenom : str = None # 12
    nom : str = None # 13
    adresse_ligne2 : str = None # 20
    adresse_ligne3 : str = None # 21
    adresse_voie : str = None # 22
    adresse_ligne5 : str = None # 23
    code_postal : str = None # 24
    localite : str = None # 25
    pays : str = None # 26


@dataclass
class RIB(DocFields):
    """Type 03 — Relevé d'Identité Bancaire"""
    qualite_nom_prenom : str  = None   # 30 - O
    code_iban  : str  = None   # 31 - O
    code_bic : str  = None   # 32 - O


@dataclass
class AvisImpotRevenu(DocFields):
    """Type 04 — Avis d'impôt sur les revenus"""
    nombre_parts : str  = None   # 43 - O
    reference_avis : str  = None   # 44 - O
    annee_revenus : str  = None   # 45 - O
    declarant1 : str  = None   # 46 - O
    numero_fiscal_d1 : str  = None   # 47 - O
    date_mise_recouvrement : str  = None   # 4A - O


@dataclass
class BulletinSalaire(DocFields):
    """Type 06 — Bulletin de salaire"""
    adresse_ligne1: str = None  # 10 - O*
    titre_personne: str = None  # 11 - O*
    prenom: str = None  # 12 - O*
    nom: str = None  # 13 - O*
    siret_employeur: str = None  # 50 - O
    debut_periode: str = None  # 53 - O
    fin_periode: str = None  # 54 - O
    date_debut_contrat: str = None  # 55 - O
    salaire_net_imposable: str = None  # 58 - O
    cumul_salaire_net: str = None  # 59 - O


# ====================
# On fait ensuite un mapping de tous les documents pouvant être vérifié
DOC_TYPE_MAP = {
    "00": JustificatifDomicile,
    "03" : RIB,
    "04" : AvisImpotRevenu,
    "06" : BulletinSalaire,
}


# ===================
# Puis on instancie les différents paramètres de résultat

@dataclass
class VerificationResult:
    statut : VerificationStatus
    message : str
    extraction_ok : Optional[bool] = None
    crypto_ok : Optional[bool] = None
    coherence_ok : Optional[bool] = None
    champs : Optional[DocFields] = None
    details : Optional[dict] = None


## 2. Extraction OCR et DataMatrix

Une fois avoir référencé les différents types documents, on s'occupe du traitement du document uploadé. On peut recevoir plusieurs types de documents (PDF ou image). Il faudra ainsi faire les tâches suivantes : 

1. Chargement et normalisation des documents (PDF / images)
2. Prétraitement des images
3. OCR avec Tesseract
4. Détection des codes 2D (DataMatrix / Barcode)
5. Traitements spécifiques mobile
6. Consolidation multi-pages

### Dépendances

Bibliothèques utilisées :
- pymupdf
- opencv-python
- numpy
- zxing-cpp
- pytesseract


In [ ]:
import pymupdf
import cv2
from pathlib import Path
import numpy as np
import zxingcpp
import pytesseract
import logging
import platform

### Configuration et constantes

Définition des extensions supportées, langues OCR et exceptions métier.

In [ ]:
# Constantes
EXTENSIONS_IMAGE = {".jpg", ".jpeg", ".png", ".tiff", ".bmp"}
MAX_FILE_SIZE_MB = 20
OCR_LANG_PRIMARY = "fra"
OCR_LANG_EXTENDED = "fra+eng+deu+spa+ita"
OCR_CONFIG = "--psm 6"

# Exceptions
class ExtractionError(Exception): pass
class InvalidImageError(ExtractionError): pass
class DataMatrixNotFoundError(ExtractionError): pass
class MultipleDataMatrixFoundError(ExtractionError): pass

### Chargement et normalisation des documents

Cette fonction :
- vérifie l'existence du fichier ;
- contrôle la taille ;
- convertit les PDF en images ;
- charge les images compatibles.

In [ ]:
def normalisation_document(file_path: Path):
    file_path = Path(file_path)

    if not file_path.exists():
        raise ExtractionError(f"Fichier introuvable: {file_path}")

    # vérification de la taille du fichier (Sécurité serveur Flask)
    if file_path.stat().st_size > MAX_FILE_SIZE_MB * 1024 * 1024:
        raise ExtractionError("Fichier trop volumineux")

    # gestion des pdf
    if file_path.suffix.lower() == ".pdf":
        doc = pymupdf.open(file_path)
        pages = []

        for i, page in enumerate(doc):
            pix = page.get_pixmap(dpi=350)  # pdf vers pixels bitmap 350 DPI
            
            #conversion vers array NumPy et reconstruction matrice image (longueurxlargeur x channels)
            img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)

            #conversion couleur sécurisée pour OpenCV
            if pix.n == 4:
                img = cv2.cvtColor(img, cv2.COLOR_RGBA2BGR)
            elif pix.n == 3:
                img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
            else:
                img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
            pages.append(img)
            logger.debug(f"Page {i+1} rasterisée")

        doc.close()
        return pages

    # Vérification des extensions valides
    if file_path.suffix.lower() in EXTENSIONS_IMAGE:
        img = cv2.imread(str(file_path))
        if img is None:
            raise InvalidImageError("Image illisible")
        return [img]

    raise InvalidImageError("Extension non supportée")

### Prétraitement d'image

Étapes :
- niveaux de gris ;
- amélioration du contraste (CLAHE) ;
- réduction du bruit ;
- binarisation adaptative.

In [ ]:
def traitement_image(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) #convertir en niveau de gris

    # augmentation contraste locale
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    gray = clahe.apply(gray)

    # réduction bruit
    blur = cv2.GaussianBlur(gray, (3, 3), 0)

    binary = cv2.adaptiveThreshold(
        blur,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        31,
        2)

    return gray, binary

### OCR

Extraction du texte via Tesseract avec mécanisme de repli.

In [ ]:
def extract_text(gray: np.ndarray, binary: np.ndarray):
    """
    Extraction du texte d'une image à l'aide de pytesseract 
    dans un format avec plusieurs langues principales. Fallback en binaire 
    si rien n'est détecté
    """
    txt = pytesseract.image_to_string(gray, lang=OCR_LANG_PRIMARY, config=OCR_CONFIG)

    if not txt.strip():
        txt = pytesseract.image_to_string(
            binary,
            lang=OCR_LANG_EXTENDED,
            config=OCR_CONFIG)

    return txt.strip()


### Détection des codes 2D

Pipeline principal de lecture DataMatrix / Barcode :
- amélioration de l'image ;
- rotations ;
- binarisation ;
- déduplication.

In [ ]:
def extract_2d_code(gray: np.ndarray, binary: np.ndarray, original: np.ndarray = None):
    """
    Extraction des 2D codes détectés dans une image, principalement le document original.
    Test fait sur plusieurs représentation de l'image, plusieurs fallback si besoin
    """
    def try_decode(img):
        try:
            decoded = zxingcpp.read_barcodes(img)
            return [r.text.strip() for r in decoded if r.text]
        except Exception:
            return []

    def enhance(img):
        # upscale
        up = cv2.resize(img, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)

        # contraste fort
        clahe = cv2.createCLAHE(3.0, (8, 8))
        gray_img = cv2.cvtColor(up, cv2.COLOR_BGR2GRAY)
        gray_img = clahe.apply(gray_img)

        # sharpen (important sur jepg flou)
        kernel = np.array([[0,-1,0],
                           [-1,5,-1],
                           [0,-1,0]])
        sharp = cv2.filter2D(gray_img, -1, kernel)
        return sharp

    def binarize(img):
        return cv2.adaptiveThreshold(
            img,
            255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY,
            31,
            2)

    candidates = []

    # base
    candidates.append(binary)
    candidates.append(gray)

    # enhanced
    candidates.append(enhance(original if original is not None else cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)))

    # binarisation
    candidates.append(binarize(gray))

    # fallback rotation
    def rotations(img):
        return [
            img,
            cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE),
            cv2.rotate(img, cv2.ROTATE_180),
            cv2.rotate(img, cv2.ROTATE_90_COUNTERCLOCKWISE)]

    all_codes = []

    for c in candidates:
        for r in rotations(c):
            codes = try_decode(r)
            if codes:
                all_codes.extend(codes)

    # dedup
    unique = list(set(all_codes))

    if not unique:
        return None

    return unique[0]


def detect_input_type(img):
    """
    Classe l'image pour adaper les traitements (surtout pour photo avec téléphone ou screenshot
    par exemple, recherche le flou, ratio, résolution)
    """
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    h, w = img.shape[:2]
    blur = cv2.Laplacian(gray, cv2.CV_64F).var()

    # screenshot = image propre + ratio typique écran
    aspect = w / h

    if blur > 200 and 0.4 < aspect < 0.7:
        return "screenshot"

    if blur < 80 or h < 1200 or w < 900:
        return "mobile"

    return "document"


def normalize_mobile_image(img):
    """
    Nettoyage spécifique smartphone :
    - contraste fort
    - réduction bruit capteur / compression JPEG
    """
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # correction contraste agressive
    clahe = cv2.createCLAHE(clipLimit=3.5, tileGridSize=(8,8))
    gray = clahe.apply(gray)

    # réduction bruit type JPEG / capteur
    gray = cv2.bilateralFilter(gray, 7, 50, 50)

    return gray


def auto_deskew(img):
    """
    Correction de perspective si document incliné grâce à la détection des contours, 
    calcul du rectangle minimal englobant, transformation perspective
    """
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    edges = cv2.Canny(gray, 50, 150)

    pts = np.column_stack(np.where(edges > 0))

    if len(pts) < 200:
        return img

    rect = cv2.minAreaRect(pts)
    box = cv2.boxPoints(rect)
    box = np.float32(box)

    w = int(rect[1][0])
    h = int(rect[1][1])

    if w == 0 or h == 0:
        return img

    dst = np.array([[0,0],[w,0],[w,h],[0,h]], dtype=np.float32)

    M = cv2.getPerspectiveTransform(box, dst)

    return cv2.warpPerspective(img, M, (w, h))


def detect_mobile_rois(img):
    """
    Découpe de l'image pour trouver les zones utiles.
    """
    h, w = img.shape[:2]

    scales = [
        (0, 1, 0, 1),# full
        (0.3, 1, 0, 1), # bottom half
        (0.2, 0.8, 0.2, 0.8), # center crop
        (0, 0.6, 0, 1)] # top region (rare fallback)

    rois = []

    for y1, y2, x1, x2 in scales:
        roi = img[
            int(h*y1):int(h*y2),
            int(w*x1):int(w*x2)]
        
        rois.append(roi)

    return rois


def generate_variants(img):
    """
    Génère plusieurs versions de l’image pour améliorer la détection de code.
    """
    variants = []

    gray = normalize_mobile_image(img)

    variants.append(gray)

    # upscale
    variants.append(cv2.resize(gray, None, fx=2, fy=2))

    # sharpen
    kernel = np.array([[0,-1,0],
                       [-1,5,-1],
                       [0,-1,0]])

    variants.append(cv2.filter2D(gray, -1, kernel))

    # threshold adaptatif
    variants.append(cv2.adaptiveThreshold(
        gray, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        31, 2))

    return variants


def extract_2d_mobile(img):
    """
    Pipeline DataMatrix mobile
    """
    img = auto_deskew(img)
    rois = detect_mobile_rois(img)
    all_codes = []

    for roi in rois:

        variants = generate_variants(roi)

        for v in variants:

            # rotations smartphone
            for r in [v,
                cv2.rotate(v, cv2.ROTATE_90_CLOCKWISE),
                cv2.rotate(v, cv2.ROTATE_180),
                cv2.rotate(v, cv2.ROTATE_90_COUNTERCLOCKWISE)
            ]:

                try:
                    results = zxingcpp.read_barcodes(r)

                    for res in results:
                        if res.text:
                            all_codes.append(res.text.strip())

                except:
                    continue

    if not all_codes:
        return None

    from collections import Counter
    return Counter(all_codes).most_common(1)[0][0]


def extract_2d_router(img, gray, binary):
    """
    Choisit la stratégie de lecture de code selon le type d’image.
    """
    mode = detect_input_type(img)

    if mode == "mobile":
        return extract_2d_mobile(img)

    if mode == "screenshot":
        up = cv2.resize(img, None, fx=2, fy=2)

        codes = zxingcpp.read_barcodes(up)
        codes = [c.text for c in codes if c.text]

        return codes[0] if codes else None

    return extract_2d_code(gray, binary, original=img)

### Extraction d'une page

Combine OCR + détection du code 2D.

In [ ]:
def extract_page(img):
    """
    Prendre une image et retourner les données brutes extraites 
    (ocr texte, extraction code et statut de validation)
    """
    if img is None:
        raise InvalidImageError("Image None")

    gray, binary = traitement_image(img)

    text = extract_text(gray, binary)

    try:
        code = extract_2d_router(img, gray, binary)
        if code is None:
            error = "missing_code"
        else:
            error = "ok"

    except DataMatrixNotFoundError:
        code = None
        error = "missing_code"
    except MultipleDataMatrixFoundError:
        code = None
        error = "multiple_codes"

    return {
        "code_2d": code,
        "text": text,
        "page_error": error}


## Extraction d'un document complet

Gestion multi-pages et consolidation des résultats.

In [ ]:
def extract_document(file_path):
    """
    Extraction complète du document (même si plusieurs pages) :
    conversion pdf/images, traitement par page, consolidation des codes
    """
    pages = normalisation_document(file_path)
    results = []
    all_codes = []

    for i, page in enumerate(pages, 1):

        data = extract_page(page)

        results.append({
            "page": i,
            **data
        })

        if data["code_2d"]:
            if data["code_2d"] not in all_codes:
                all_codes.append(data["code_2d"])


    # aucun code du tout => ok mais document incomplet
    if not all_codes:
        return {
            "pages": results,
            "document_validation": {
                "unique_codes": [],
                "status": "no_2d_code_found"
            }
        }

    # 1 code global => cas standard fiscal
    if len(all_codes) == 1:
        return {
            "pages": results,
            "document_validation": {
                "unique_codes": all_codes,
                "status": "single_code_ok"
            }
        }

    # plusieurs codes différents => on les accepte tous mais on signale l’anomalie sans exception
    return {
        "pages": results,
        "document_validation": {
            "unique_codes": all_codes,
            "status": "multiple_codes_detected_but_accepted"
        }
    }

### Exemple d'utilisation

In [3]:
file_path = 'document.pdf'

# result = extract_document(file_path)

Ainsi, grâce à cette étape, on peut extraite tous le texte brut et le DataMatrix en brut qui permettra ensuite de le parser, et de comparer **les résultats OCR avec le DataMatrix**.

Le résultat qui resort de la fonction *extract_document(file_path)* ressemble à ça :




In [1]:
example_result = {
    "pages": [
        {
            "page": 1,
            "code_2d": "DMX2025ABCD1234XYZ7890FAKECODE001",
            "text": """
SOCIÉTÉ EXEMPLE INDUSTRIES
Facture n° FAC-2025-001245

Client :
Jean DUPONT
15 avenue des Lilas
75000 Paris

Date : 15/03/2025

Montant HT : 1 250,00 €
TVA 20 % : 250,00 €
Montant TTC : 1 500,00 €
""",
            "page_error": "ok"
        },
        {
            "page": 2,
            "code_2d": "DMX2025ABCD1234XYZ7890FAKECODE002",
            "text": """
Détail des prestations

- Conseil technique : 750,00 €
- Développement : 350,00 €
- Support : 150,00 €

Total HT : 1 250,00 €
""",
            "page_error": "ok"
        },
        {
            "page": 3,
            "code_2d": None,
            "text": """
Conditions générales

Paiement à 30 jours.
Pénalités de retard applicables conformément aux
conditions contractuelles en vigueur.
""",
            "page_error": "missing_code"
        }
    ],
    "document_validation": {
        "unique_codes": [
            "DMX2025ABCD1234XYZ7890FAKECODE001",
            "DMX2025ABCD1234XYZ7890FAKECODE002"
        ],
        "status": "multiple_codes_detected_but_accepted"
    }
}